In [43]:
!pip install discord --quiet
!pip install transformers datasets huggingface_hub --quiet
!pip install keybert --quiet

In [45]:
import asyncio
import nest_asyncio
import discord
from discord.ext import commands
import pandas as pd
from datetime import datetime, timedelta, timezone
import re
from transformers import pipeline
from huggingface_hub import login
from keybert import KeyBERT
import textwrap
from google.cloud import bigquery
from google.oauth2 import service_account
from google.auth import default
from pandas_gbq import to_gbq
import textwrap
from transformers import pipeline
from keybert import KeyBERT

### Discord Import

In [ ]:
# bot ID
bot_id = os.environ.get('DISCORD_BOT_TOKEN')  # set your token as an environment variable

intents = discord.Intents.default()
intents.message_content = True
intents.guilds = True
intents.messages = True
bot = commands.Bot(command_prefix="$", intents=intents)

nest_asyncio.apply()

all_messages = []

In [ ]:
class MyClient(discord.Client):
    async def on_ready(self):
        print(f'Logged in as {self.user.name}')

        guild_id = 666344617536520207
        channel_id = 842090288583278602
        guild = self.get_guild(guild_id)

        if not guild:
            print(f"Guild with ID {guild_id} not found.")
            await self.close()
            return

        general_channel = guild.get_channel(channel_id)

        if general_channel:
            print(f'Found channel: #{general_channel.name} (ID: {general_channel.id})')

            one_hour_ago = datetime.now(timezone.utc) - timedelta(hours=24)

            async for message in general_channel.history(limit=None, oldest_first=False):
                if message.created_at >= one_hour_ago:
                    all_messages.append({
                        'Timestamp': message.created_at.strftime('%Y-%m-%d %H:%M:%S'),
                        'Author': str(message.author),
                        'Content': message.content
                    })
                else:
                    break  # Stop early if messages are older than the cutoff
        else:
            print('Channel not found')

        await self.close()

In [ ]:
# start the bot and collect messages
client = MyClient(intents=intents)
await client.start(bot_id)

# create the DataFrame after bot finishes
df = pd.DataFrame(columns=['Timestamp', 'Author', 'Content'])  # initialize with correct columns
df = pd.DataFrame(all_messages)
df.head()

Logged in as analytics
Found channel: #⏳general (ID: 842090288583278602)


,Timestamp,Author,Content
0,2025-06-10 15:03:27,sourav_48784,hey
1,2025-06-10 14:56:26,lococolo,Good morning everyone
2,2025-06-10 14:56:08,lococolo,"Hi Kouli, please use the <#1340656899431071744..."
3,2025-06-10 12:50:54,sil3nt_5496,CMC is not an exchange.\nhttps : // coinmarket...
4,2025-06-10 12:27:04,shun3636,CMC is not our platform. And if you receive it...


### Text Cleaning

In [ ]:
# combine message contents into one string and clean in
def clean_discord_text(text):
    # Remove mentions like <@123456789>
    text = re.sub(r'<@!?[0-9]+>', '', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Remove markdown formatting (*, _, ~)
    text = re.sub(r'[*_~`]', '', text)

    # Remove custom emoji codes like <:emoji_name:1234567890>
    text = re.sub(r'<:[a-zA-Z0-9_]+:[0-9]+>', '', text)

    # Remove standard emojis (basic Unicode emoji pattern)
    text = re.sub(r'[\U00010000-\U0010ffff]', '', text)

    # Remove extra spaces and newlines
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
# join cleaned text
df['clean_content'] = df['Content'].dropna().apply(clean_discord_text)
text = "\n".join(df["clean_content"].tolist())

def chunk_text(text, max_chars=3000):
  return textwrap.wrap(text, max_chars)

chunks = chunk_text(text)
print(f'Total Chunks: {len(chunks)}')

Total Chunks: 4


In [44]:
# join cleaned text
df['clean_content'] = df['Content'].dropna().apply(clean_discord_text)
text = "\n".join(df["clean_content"].tolist())

def chunk_text(text, max_chars=3000):
  return textwrap.wrap(text, max_chars)

chunks = chunk_text(text)

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=0,               # GPU
    batch_size=4,           # tweak to fit your GPU memory
)

# ====== 2. Summarize each chunk ======
summaries = []
for chunk in chunks:
    # adjust max_length/min_length for your use‐case
    summary = summarizer(chunk, max_length=100, min_length=30, do_sample=False)
    summaries.append(summary[0]["summary_text"])

# Merge summaries into one overall summary
final_summary = summarizer(" ".join(summaries), max_length=150, min_length=50, do_sample=False)[0]["summary_text"]

print("=== Overall Summary ===")
print(final_summary)

Device set to use cpu
Device set to use cpu


=== Overall Summary ===
Daily rewards for VIP levels 1-4 are 10 times lower than expected. Only VIP levels 5 and above are normal. CMC is not an exchange. Titles can be crafted in Time Wardens. If the rewards are transferred by the Open Loot team, they will be delivered to your Open Loot account.


In [ ]:
from transformers import AutoTokenizer

# 4a. Run sentiment on each message
#    (Better than chunking, so we get finer granularity)
df["sentiment"] = [
    s["label"] for s in sentiment_analyzer(df["clean_content"].tolist())
]

# 4b. Pull out the negative messages
negative_df = df[df["sentiment"] == "NEGATIVE"]
neg_text = " ".join(negative_df["clean_content"].tolist())
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
inputs = tokenizer(
    neg_text,
    return_tensors="pt",
    max_length=1024,      # BART's actual max input length
    truncation=True
)

# Move to GPU if available
inputs = {k: v.cuda() for k, v in inputs.items()} if summarizer.device == 0 else inputs

# Then summarize
summary_ids = summarizer.model.generate(**inputs, max_length=80, min_length=20, do_sample=False)
neg_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)


In [ ]:
neg_summary

'CMC is not an exchange. If the rewards are transferred by the Open Loot team, they will be delivered to your Open Loot account. The daily rewards for VIP levels 1-4 are 10 times lower than expected. Only VIP levels 5 and above are normal.'